Phase 9 — Model Improvement and Validation

Step 9.1 — Restore and Verify Saved Data and Model

In [3]:

# ============================================================
# PHASE 9 — STEP 9.1: RESTORE AND VERIFY SAVED DATA AND MODEL
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# 1. Project paths
PROJECT_DIR = Path(r"D:\smart-cybersecurity-ml")
DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_PATH = PROJECT_DIR / "models" / "final_xgboost_model.pkl"
RESULTS_DIR = PROJECT_DIR / "results" / "phase9"
REPORTS_DIR = PROJECT_DIR / "reports" / "phase9"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load encoded datasets and labels
X_train = pd.read_csv(DATA_DIR / "X_train_encoded.csv")
X_test = pd.read_csv(DATA_DIR / "X_test_encoded.csv")

y_train = np.load(DATA_DIR / "y_train_encoded.npy")
y_test = np.load(DATA_DIR / "y_test_encoded.npy")

# 3. Load saved preprocessing outputs
X_test_scaled = np.load(DATA_DIR / "X_test_scaled.npy")
X_train_resampled = np.load(DATA_DIR / "X_train_resampled.npy")
y_train_resampled = np.load(DATA_DIR / "y_train_resampled.npy")

feature_encoders = joblib.load(DATA_DIR / "feature_encoders.pkl")
target_encoder = joblib.load(DATA_DIR / "target_encoder.pkl")
scaler = joblib.load(DATA_DIR / "scaler.pkl")

# 4. Load the saved model
xgb_model = joblib.load(MODEL_PATH)

# Recover known original class labels
class_names = target_encoder.classes_

# 5. Print dataset information
print("=" * 75)
print("PHASE 9 — STEP 9.1: DATA AND MODEL VERIFICATION")
print("=" * 75)

print("\n--- Encoded Data ---")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\n--- Saved Arrays ---")
print("X_test_scaled:", X_test_scaled.shape)
print("X_train_resampled:", X_train_resampled.shape)
print("y_train_resampled:", y_train_resampled.shape)

print("\n--- Model Information ---")
print("Model type:", type(xgb_model).__name__)
print("Known classes:", len(class_names))
print("Model input features:",
      getattr(xgb_model, "n_features_in_", "Not available"))

# 6. Integrity checks
assert len(X_train) == len(y_train), \
    "Training feature/label lengths do not match."

assert len(X_test) == len(y_test), \
    "Test feature/label lengths do not match."

assert X_train.shape[1] == X_test.shape[1], \
    "Training and test feature counts do not match."

assert X_test_scaled.shape[0] == len(y_test), \
    "Scaled test data does not match test label count."

assert X_train_resampled.shape[0] == len(y_train_resampled), \
    "Resampled training feature/label lengths do not match."

assert X_train.columns.tolist() == X_test.columns.tolist(), \
    "Training and test feature columns/order differ."

assert hasattr(xgb_model, "predict"), \
    "Loaded model does not support prediction."

model_features = getattr(xgb_model, "n_features_in_", None)
if model_features is not None:
    assert model_features == X_test.shape[1], \
        "Model feature count does not match encoded test data."

# 7. Save verification summary
summary = [
    "PHASE 9 — STEP 9.1: DATA AND MODEL VERIFICATION",
    f"X_train shape: {X_train.shape}",
    f"y_train shape: {y_train.shape}",
    f"X_test shape: {X_test.shape}",
    f"y_test shape: {y_test.shape}",
    f"X_test_scaled shape: {X_test_scaled.shape}",
    f"Model type: {type(xgb_model).__name__}",
    f"Known classes: {len(class_names)}",
    "Basic integrity checks: PASSED"
]

summary_path = RESULTS_DIR / "phase9_step9_1_verification.txt"
summary_path.write_text("\n".join(summary), encoding="utf-8")

print("\nAll basic integrity checks passed.")
print("Verification report saved to:", summary_path)


PHASE 9 — STEP 9.1: DATA AND MODEL VERIFICATION

--- Encoded Data ---
X_train: (118783, 41)
y_train: (118783,)
X_test: (29696, 41)
y_test: (29696,)

--- Saved Arrays ---
X_test_scaled: (29696, 41)
X_train_resampled: (135441, 41)
y_train_resampled: (135441,)

--- Model Information ---
Model type: XGBClassifier
Known classes: 28
Model input features: 41

All basic integrity checks passed.
Verification report saved to: D:\smart-cybersecurity-ml\results\phase9\phase9_step9_1_verification.txt



## Step 9.2 — Data Leakage Investigation


In [4]:

# ============================================================
# PHASE 9 — STEP 9.2: DATA LEAKAGE INVESTIGATION
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 75)
print("PHASE 9 — STEP 9.2: DATA LEAKAGE INVESTIGATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Check whether target labels appear in feature columns
# ------------------------------------------------------------

print("\n--- 1. Target Column Check ---")

possible_target_columns = [
    "label", "labels", "class", "target", "attack_type"
]

found_target_columns = [
    col for col in X_train.columns
    if col.lower() in possible_target_columns
]

print("Possible target columns in features:", found_target_columns)

if found_target_columns:
    print("WARNING: Investigate these columns before training.")
else:
    print("No obvious target column found in feature names.")

# ------------------------------------------------------------
# 2. Check duplicate rows within each dataset
# ------------------------------------------------------------

print("\n--- 2. Duplicate Row Check ---")

train_duplicates = int(X_train.duplicated().sum())
test_duplicates = int(X_test.duplicated().sum())

print("Duplicate feature rows in training:", train_duplicates)
print("Duplicate feature rows in test:", test_duplicates)

# ------------------------------------------------------------
# 3. Check exact feature-row overlap across train and test
# ------------------------------------------------------------

print("\n--- 3. Train-Test Feature Overlap ---")

# Hash rows to avoid concatenating the full datasets.
# This checks exact feature-row matches, not semantic similarity.
train_hashes = pd.util.hash_pandas_object(
    X_train, index=False
)

test_hashes = pd.util.hash_pandas_object(
    X_test, index=False
)

train_hash_set = set(train_hashes.tolist())
test_hash_set = set(test_hashes.tolist())

overlap_hashes = train_hash_set.intersection(test_hash_set)

print("Unique training feature rows:", len(train_hash_set))
print("Unique test feature rows:", len(test_hash_set))
print("Matching row hashes across datasets:", len(overlap_hashes))

# Confirm matches directly for rows with matching hashes,
# so hash collisions alone are not counted as duplicate rows.
train_rows_by_hash = {}

for idx, row_hash in enumerate(train_hashes.tolist()):
    train_rows_by_hash.setdefault(row_hash, []).append(idx)

verified_overlap = 0

for test_idx, row_hash in enumerate(test_hashes.tolist()):
    if row_hash not in train_rows_by_hash:
        continue

    test_row = X_test.iloc[test_idx]

    for train_idx in train_rows_by_hash[row_hash]:
        if X_train.iloc[train_idx].equals(test_row):
            verified_overlap += 1
            break

print("Verified test rows matching a training feature row:",
      verified_overlap)

# ------------------------------------------------------------
# 4. Check train/test feature schema
# ------------------------------------------------------------

print("\n--- 4. Feature Schema Check ---")

same_columns = X_train.columns.tolist() == X_test.columns.tolist()

print("Same feature names and order:", same_columns)
print("Training feature count:", X_train.shape[1])
print("Test feature count:", X_test.shape[1])

# ------------------------------------------------------------
# 5. Save findings
# ------------------------------------------------------------

RESULTS_DIR = Path(r"D:\smart-cybersecurity-ml\results\phase9")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

report_path = RESULTS_DIR / "phase9_step9_2_leakage_investigation.txt"

report_lines = [
    "PHASE 9 — STEP 9.2: DATA LEAKAGE INVESTIGATION",
    f"Target-like feature columns: {found_target_columns}",
    f"Training duplicate feature rows: {train_duplicates}",
    f"Test duplicate feature rows: {test_duplicates}",
    f"Unique training feature rows: {len(train_hash_set)}",
    f"Unique test feature rows: {len(test_hash_set)}",
    f"Matching row hashes across datasets: {len(overlap_hashes)}",
    f"Verified overlapping test rows: {verified_overlap}",
    f"Same feature names and order: {same_columns}",
    "",
    "Note: These checks do not prove that the complete pipeline is leakage-free."
]

report_path.write_text("\n".join(report_lines), encoding="utf-8")

print("\nLeakage investigation saved to:", report_path)
print("Step 9.2 diagnostic checks completed.")


PHASE 9 — STEP 9.2: DATA LEAKAGE INVESTIGATION

--- 1. Target Column Check ---
Possible target columns in features: []
No obvious target column found in feature names.

--- 2. Duplicate Row Check ---
Duplicate feature rows in training: 10144
Duplicate feature rows in test: 589

--- 3. Train-Test Feature Overlap ---
Unique training feature rows: 108639
Unique test feature rows: 29107
Matching row hashes across datasets: 5089
Verified test rows matching a training feature row: 5094

--- 4. Feature Schema Check ---
Same feature names and order: True
Training feature count: 41
Test feature count: 41

Leakage investigation saved to: D:\smart-cybersecurity-ml\results\phase9\phase9_step9_2_leakage_investigation.txt
Step 9.2 diagnostic checks completed.


Step 9.2.1 — Verify whether overlapping rows have matching labels

In [5]:

# ============================================================
# PHASE 9 — STEP 9.2.1: VERIFY OVERLAPPING ROW LABELS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 75)
print("PHASE 9 — STEP 9.2.1: OVERLAPPING ROW LABEL VERIFICATION")
print("=" * 75)

# 1. Build a mapping from each training feature row to its labels.
train_hashes = pd.util.hash_pandas_object(
    X_train, index=False
).to_numpy()

test_hashes = pd.util.hash_pandas_object(
    X_test, index=False
).to_numpy()

train_labels = np.asarray(y_train).ravel()
test_labels = np.asarray(y_test).ravel()

# Hashes are used to find candidate matches.
# Exact feature equality is checked afterward.
train_lookup = {}

for i, row_hash in enumerate(train_hashes):
    train_lookup.setdefault(row_hash, []).append(i)

overlapping_test_rows = 0
same_label_rows = 0
different_label_rows = 0
ambiguous_training_patterns = set()

for test_i, row_hash in enumerate(test_hashes):

    candidate_indices = train_lookup.get(row_hash, [])

    if not candidate_indices:
        continue

    test_row = X_test.iloc[test_i]
    exact_matches = []

    for train_i in candidate_indices:
        if X_train.iloc[train_i].equals(test_row):
            exact_matches.append(train_i)

    if not exact_matches:
        continue

    overlapping_test_rows += 1

    matching_train_labels = {
        train_labels[i].item()
        if isinstance(train_labels[i], np.generic)
        else train_labels[i]
        for i in exact_matches
    }

    if len(matching_train_labels) > 1:
        ambiguous_training_patterns.add(tuple(
            X_train.iloc[exact_matches[0]].tolist()
        ))

    actual_test_label = test_labels[test_i]
    if isinstance(actual_test_label, np.generic):
        actual_test_label = actual_test_label.item()

    if actual_test_label in matching_train_labels:
        same_label_rows += 1
    else:
        different_label_rows += 1

# 2. Print results
print("\n--- Overlap Label Analysis ---")
print("Test rows with exact training-feature matches:",
      overlapping_test_rows)
print("Overlapping rows with a matching label:",
      same_label_rows)
print("Overlapping rows with no matching training label:",
      different_label_rows)
print("Training feature patterns associated with multiple labels:",
      len(ambiguous_training_patterns))

# 3. Save results
RESULTS_DIR = Path(r"D:\smart-cybersecurity-ml\results\phase9")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

report_path = RESULTS_DIR / "phase9_step9_2_1_overlap_label_check.txt"

report = [
    "PHASE 9 — STEP 9.2.1: OVERLAPPING ROW LABEL VERIFICATION",
    f"Exact overlapping test rows: {overlapping_test_rows}",
    f"Overlapping rows with a matching label: {same_label_rows}",
    f"Overlapping rows with no matching training label: {different_label_rows}",
    f"Training patterns with multiple labels: {len(ambiguous_training_patterns)}",
    "",
    "Interpret these results in the context of the dataset and split procedure."
]

report_path.write_text("\n".join(report), encoding="utf-8")

print("\nReport saved to:", report_path)
print("Step 9.2.1 completed.")


PHASE 9 — STEP 9.2.1: OVERLAPPING ROW LABEL VERIFICATION

--- Overlap Label Analysis ---
Test rows with exact training-feature matches: 5094
Overlapping rows with a matching label: 5086
Overlapping rows with no matching training label: 8
Training feature patterns associated with multiple labels: 2

Report saved to: D:\smart-cybersecurity-ml\results\phase9\phase9_step9_2_1_overlap_label_check.txt
Step 9.2.1 completed.



## Step 9.2.2 — Inspect Conflicting Labels in Overlapping Rows


In [6]:

# ============================================================
# PHASE 9 — STEP 9.2.2: INSPECT CONFLICTING OVERLAPPING ROWS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 75)
print("PHASE 9 — STEP 9.2.2: CONFLICTING LABEL INSPECTION")
print("=" * 75)

train_labels = np.asarray(y_train).ravel()
test_labels = np.asarray(y_test).ravel()

train_hashes = pd.util.hash_pandas_object(
    X_train, index=False
).to_numpy()

test_hashes = pd.util.hash_pandas_object(
    X_test, index=False
).to_numpy()

# Index training rows by hash
train_lookup = {}

for train_i, row_hash in enumerate(train_hashes):
    train_lookup.setdefault(row_hash, []).append(train_i)

conflicts = []

for test_i, row_hash in enumerate(test_hashes):
    candidate_indices = train_lookup.get(row_hash, [])

    if not candidate_indices:
        continue

    test_row = X_test.iloc[test_i]
    exact_matches = [
        train_i
        for train_i in candidate_indices
        if X_train.iloc[train_i].equals(test_row)
    ]

    if not exact_matches:
        continue

    matching_labels = {
        train_labels[i].item()
        if isinstance(train_labels[i], np.generic)
        else train_labels[i]
        for i in exact_matches
    }

    actual_label = test_labels[test_i]
    if isinstance(actual_label, np.generic):
        actual_label = actual_label.item()

    if actual_label not in matching_labels:
        conflicts.append({
            "test_row_index": test_i,
            "test_label_encoded": actual_label,
            "matching_train_labels_encoded": sorted(
                matching_labels, key=str
            ),
            "matching_training_rows": len(exact_matches)
        })

# Display conflicting rows
conflict_df = pd.DataFrame(conflicts)

print("\nNumber of conflicting test rows:", len(conflict_df))

if not conflict_df.empty:
    print("\n--- Conflicting Row Details ---")
    print(conflict_df.to_string(index=False))

    # Add original class names when the encoder supports inverse_transform
    if hasattr(target_encoder, "inverse_transform"):
        print("\n--- Decoded Class Names ---")

        for _, row in conflict_df.iterrows():
            test_class = target_encoder.inverse_transform(
                np.asarray([row["test_label_encoded"]])
            )[0]

            train_classes = target_encoder.inverse_transform(
                np.asarray(row["matching_train_labels_encoded"])
            )

            print(
                f"Test row {row['test_row_index']}: "
                f"test label = {test_class}; "
                f"matching training labels = {list(train_classes)}"
            )
else:
    print("No conflicting rows found.")

# Save findings
results_dir = Path(r"D:\smart-cybersecurity-ml\results\phase9")
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "phase9_step9_2_2_conflicting_rows.csv"
conflict_df.to_csv(output_path, index=False)

print("\nConflicting-row report saved to:", output_path)
print("Step 9.2.2 completed.")


PHASE 9 — STEP 9.2.2: CONFLICTING LABEL INSPECTION

Number of conflicting test rows: 8

--- Conflicting Row Details ---
 test_row_index  test_label_encoded matching_train_labels_encoded  matching_training_rows
           5696                  15                           [8]                       1
          10247                  23                          [15]                       1
          11003                  15                          [16]                       1
          17649                   7                          [14]                       1
          19388                  15                           [8]                       1
          20825                  14                           [7]                       1
          21138                  15                          [23]                       1
          29581                  25                          [15]                       1

--- Decoded Class Names ---
Test row 5696: test label = normal; match

Phase 9 — Step 9.3: Rare-Class Error Analysis


## Step 9.3 — Rare-Class Error Analysis
### Per-Class Precision, Recall, F1-Score, and Support


In [7]:

# ============================================================
# PHASE 9 — STEP 9.3: RARE-CLASS ERROR ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    classification_report,
    precision_recall_fscore_support
)

print("=" * 75)
print("PHASE 9 — STEP 9.3: RARE-CLASS ERROR ANALYSIS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Predict using the existing saved model
# ------------------------------------------------------------

y_test_actual = np.asarray(y_test).ravel()

# Use the saved scaled test features used in the previous evaluation.
y_test_predicted = np.asarray(
    xgb_model.predict(X_test_scaled)
).ravel()

assert len(y_test_actual) == len(y_test_predicted), (
    "Prediction count does not match test label count."
)

print("\nTest samples:", len(y_test_actual))
print("Predictions:", len(y_test_predicted))

# ------------------------------------------------------------
# 2. Calculate per-class metrics
# ------------------------------------------------------------

observed_labels = np.unique(
    np.concatenate([y_test_actual, y_test_predicted])
)

precision, recall, f1, support = precision_recall_fscore_support(
    y_test_actual,
    y_test_predicted,
    labels=observed_labels,
    zero_division=0
)

# Decode numeric labels into original class names
decoded_names = target_encoder.inverse_transform(
    observed_labels.astype(int)
)

results_df = pd.DataFrame({
    "class_id": observed_labels,
    "class_name": decoded_names,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "support": support
})

results_df["predicted_count"] = [
    int(np.sum(y_test_predicted == label))
    for label in observed_labels
]

# ------------------------------------------------------------
# 3. Identify classes with zero recall
# ------------------------------------------------------------

zero_recall_df = results_df[results_df["recall"] == 0].copy()

print("\n--- Classes With Zero Recall ---")

if zero_recall_df.empty:
    print("No classes with zero recall.")
else:
    print(
        zero_recall_df[
            ["class_name", "support", "predicted_count", "recall", "f1_score"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 4. Identify rare classes
# ------------------------------------------------------------

RARE_CLASS_THRESHOLD = 50

rare_classes_df = results_df[
    results_df["support"] < RARE_CLASS_THRESHOLD
].copy()

print(f"\n--- Rare Classes (support < {RARE_CLASS_THRESHOLD}) ---")

if rare_classes_df.empty:
    print("No classes below the rare-class threshold.")
else:
    print(
        rare_classes_df[
            ["class_name", "support", "precision", "recall", "f1_score"]
        ].sort_values("support").to_string(index=False)
    )

# ------------------------------------------------------------
# 5. Save detailed results
# ------------------------------------------------------------

RESULTS_DIR = Path(r"D:\smart-cybersecurity-ml\results\phase9")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

all_results_path = RESULTS_DIR / "phase9_step9_3_class_metrics.csv"
zero_recall_path = RESULTS_DIR / "phase9_step9_3_zero_recall_classes.csv"
rare_classes_path = RESULTS_DIR / "phase9_step9_3_rare_classes.csv"

results_df.to_csv(all_results_path, index=False)
zero_recall_df.to_csv(zero_recall_path, index=False)
rare_classes_df.to_csv(rare_classes_path, index=False)

# ------------------------------------------------------------
# 6. Print summary
# ------------------------------------------------------------

print("\n--- Summary ---")
print("Number of classes evaluated:", len(results_df))
print("Classes with zero recall:", len(zero_recall_df))
print("Rare classes:", len(rare_classes_df))

print("\nSaved reports:")
print(all_results_path)
print(zero_recall_path)
print(rare_classes_path)

print("\nStep 9.3 completed.")


PHASE 9 — STEP 9.3: RARE-CLASS ERROR ANALYSIS

Test samples: 29696
Predictions: 29696

--- Classes With Zero Recall ---
class_name  support  predicted_count  recall  f1_score
 ftp_write        2                0     0.0       0.0
  multihop        3                0     0.0       0.0
   rootkit        3                2     0.0       0.0

--- Rare Classes (support < 50) ---
     class_name  support  precision   recall  f1_score
      ftp_write        2   0.000000 0.000000  0.000000
           imap        2   1.000000 1.000000  1.000000
     loadmodule        2   1.000000 0.500000  0.666667
       multihop        3   0.000000 0.000000  0.000000
        rootkit        3   0.000000 0.000000  0.000000
           land        4   0.571429 1.000000  0.727273
     httptunnel        8   1.000000 1.000000  1.000000
buffer_overflow        9   1.000000 0.888889  0.941176
  snmpgetattack        9   0.636364 0.777778  0.700000
       mailbomb       19   1.000000 1.000000  1.000000
          saint   

Step 9.4 — Class Imbalance Analysis and Strategy


## Step 9.4 — Class Imbalance Analysis
### Training Class Distribution and Balanced Class Weights


In [9]:

# ============================================================
# PHASE 9 — STEP 9.4: CLASS IMBALANCE ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight

print("=" * 75)
print("PHASE 9 — STEP 9.4: CLASS IMBALANCE ANALYSIS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Use original training labels only
# ------------------------------------------------------------

y_train_original = np.asarray(y_train).ravel()

classes, counts = np.unique(
    y_train_original,
    return_counts=True
)

# ------------------------------------------------------------
# 2. Calculate balanced class weights from training data
# ------------------------------------------------------------

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_original
)

# ------------------------------------------------------------
# 3. Build a class distribution report
# ------------------------------------------------------------

class_names_decoded = target_encoder.inverse_transform(
    classes.astype(int)
)

imbalance_df = pd.DataFrame({
    "class_id": classes,
    "class_name": class_names_decoded,
    "training_samples": counts,
    "training_percentage": counts / len(y_train_original) * 100,
    "balanced_class_weight": weights
})

imbalance_df = imbalance_df.sort_values(
    "training_samples",
    ascending=True
).reset_index(drop=True)

print("\n--- Training Dataset Summary ---")
print("Total training samples:", len(y_train_original))
print("Number of training classes:", len(classes))
print("Smallest class size:", int(counts.min()))
print("Largest class size:", int(counts.max()))
print("Imbalance ratio:", round(counts.max() / counts.min(), 2))

print("\n--- 10 Smallest Training Classes ---")
print(
    imbalance_df[
        ["class_name", "training_samples", "balanced_class_weight"]
    ].head(10).to_string(index=False)
)

print("\n--- 10 Largest Training Classes ---")
print(
    imbalance_df[
        ["class_name", "training_samples", "balanced_class_weight"]
    ].tail(10).sort_values(
        "training_samples", ascending=False
    ).to_string(index=False)
)

# ------------------------------------------------------------
# 4. Save results
# ------------------------------------------------------------

RESULTS_DIR = Path(r"D:\smart-cybersecurity-ml\results\phase9")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

distribution_path = RESULTS_DIR / "phase9_step9_4_class_weights.csv"
imbalance_df.to_csv(distribution_path, index=False)

print("\nClass imbalance report saved to:")
print(distribution_path)

print("\nStep 9.4 completed.")


PHASE 9 — STEP 9.4: CLASS IMBALANCE ANALYSIS

--- Training Dataset Summary ---
Total training samples: 118783
Number of training classes: 28
Smallest class size: 8
Largest class size: 62870
Imbalance ratio: 7858.75

--- 10 Smallest Training Classes ---
     class_name  training_samples  balanced_class_weight
      ftp_write                 8             530.281250
           imap                11             385.659091
     loadmodule                11             385.659091
        rootkit                11             385.659091
       multihop                13             326.326923
           land                16             265.140625
     httptunnel                33             128.553030
  snmpgetattack                34             124.772059
buffer_overflow                38             111.638158
          saint                74              57.327703

--- 10 Largest Training Classes ---
 class_name  training_samples  balanced_class_weight
     normal             62870 

Step 9.5 — Compare Baseline and Class-Weighted XGBoost


## Step 9.5 — Model Improvement Using Class Weights
### Baseline XGBoost vs Class-Weighted XGBoost


In [11]:

# ============================================================
# PHASE 9 — STEP 9.5: BASELINE VS CLASS-WEIGHTED XGBOOST
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

print("=" * 75)
print("PHASE 9 — STEP 9.5: XGBOOST MODEL IMPROVEMENT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Prepare original training data
# ------------------------------------------------------------

X = X_train.copy()
y = np.asarray(y_train).ravel()

assert len(X) == len(y), "Training feature and label counts differ."
assert X.columns.tolist() == X_test.columns.tolist(), (
    "Training and test feature columns differ."
)
assert X.isna().sum().sum() == 0, (
    "Missing values found in training features."
)

# Use a stratified split so each class is represented in
# both subsets where its sample count permits.
X_fit, X_val, y_fit, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n--- Validation Split ---")
print("Model training samples:", len(X_fit))
print("Validation samples:", len(X_val))
print("Training classes:", len(np.unique(y_fit)))
print("Validation classes:", len(np.unique(y_val)))

# ------------------------------------------------------------
# 2. Create two fresh models with the saved model's parameters
# ------------------------------------------------------------

# These are new, unfitted estimators. The saved model remains unchanged.
baseline_model = clone(xgb_model)
weighted_model = clone(xgb_model)

# Use only the model-fitting subset to calculate class weights.
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_fit
)

# ------------------------------------------------------------
# 3. Train baseline XGBoost
# ------------------------------------------------------------

print("\nTraining baseline XGBoost...")

baseline_model.fit(X_fit, y_fit)

# ------------------------------------------------------------
# 4. Train class-weighted XGBoost
# ------------------------------------------------------------

print("Training class-weighted XGBoost...")

weighted_model.fit(
    X_fit,
    y_fit,
    sample_weight=sample_weights
)

# ------------------------------------------------------------
# 5. Evaluate both models on the same validation subset
# ------------------------------------------------------------

baseline_predictions = baseline_model.predict(X_val)
weighted_predictions = weighted_model.predict(X_val)

def calculate_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

baseline_metrics = calculate_metrics(y_val, baseline_predictions)
weighted_metrics = calculate_metrics(y_val, weighted_predictions)

comparison_df = pd.DataFrame(
    [baseline_metrics, weighted_metrics],
    index=["Baseline XGBoost", "Class-Weighted XGBoost"]
)

print("\n--- Validation Metrics ---")
print(comparison_df.round(4).to_string())

# ------------------------------------------------------------
# 6. Compare class-wise recall
# ------------------------------------------------------------

baseline_report = classification_report(
    y_val,
    baseline_predictions,
    output_dict=True,
    zero_division=0
)

weighted_report = classification_report(
    y_val,
    weighted_predictions,
    output_dict=True,
    zero_division=0
)

class_rows = []

for class_id in np.unique(y_val):
    key = str(class_id)

    class_rows.append({
        "class_id": int(class_id),
        "class_name": str(
            target_encoder.inverse_transform(
                np.asarray([class_id]).astype(int)
            )[0]
        ),
        "support": int(np.sum(y_val == class_id)),
        "baseline_recall": baseline_report.get(
            key, {}
        ).get("recall", 0.0),
        "weighted_recall": weighted_report.get(
            key, {}
        ).get("recall", 0.0),
        "baseline_f1": baseline_report.get(
            key, {}
        ).get("f1-score", 0.0),
        "weighted_f1": weighted_report.get(
            key, {}
        ).get("f1-score", 0.0)
    })

class_comparison_df = pd.DataFrame(class_rows)

print("\n--- Rare-Class Validation Comparison ---")
print(
    class_comparison_df[
        class_comparison_df["support"] < 50
    ].sort_values("support").round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 7. Save reports and candidate models
# ------------------------------------------------------------

RESULTS_DIR = Path(r"D:\smart-cybersecurity-ml\results\phase9")
MODELS_DIR = Path(r"D:\smart-cybersecurity-ml\models")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    RESULTS_DIR / "phase9_step9_5_validation_comparison.csv"
)

class_comparison_df.to_csv(
    RESULTS_DIR / "phase9_step9_5_classwise_comparison.csv",
    index=False
)

# Save these as experimental candidates, not final models.
import joblib

joblib.dump(
    baseline_model,
    MODELS_DIR / "phase9_baseline_xgboost_candidate.pkl"
)

joblib.dump(
    weighted_model,
    MODELS_DIR / "phase9_weighted_xgboost_candidate.pkl"
)

print("\nSaved validation comparison:")
print(RESULTS_DIR / "phase9_step9_5_validation_comparison.csv")

print("\nSaved class-wise comparison:")
print(RESULTS_DIR / "phase9_step9_5_classwise_comparison.csv")

print("\nExperimental candidate models saved.")
print("Step 9.5 completed.")


PHASE 9 — STEP 9.5: XGBOOST MODEL IMPROVEMENT

--- Validation Split ---
Model training samples: 95026
Validation samples: 23757
Training classes: 28
Validation classes: 28

Training baseline XGBoost...
Training class-weighted XGBoost...

--- Validation Metrics ---
                        accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1
Baseline XGBoost          0.9974             0.7938           0.8945        0.7938    0.8267
Class-Weighted XGBoost    0.9964             0.9057           0.9001        0.9057    0.8869

--- Rare-Class Validation Comparison ---
 class_id      class_name  support  baseline_recall  weighted_recall  baseline_f1  weighted_f1
        3       ftp_write        1           1.0000           1.0000       1.0000       1.0000
        6            imap        2           0.5000           0.5000       0.6667       0.6667
        9      loadmodule        2           0.5000           1.0000       0.6667       0.5000
       19         rootkit        2


## Step 9.6 — Validation-Based Model Selection
### Select the Best XGBoost Candidate


In [12]:

# ============================================================
# PHASE 9 — STEP 9.6: VALIDATION-BASED MODEL SELECTION
# ============================================================

from pathlib import Path
import json
import joblib
import pandas as pd

print("=" * 75)
print("PHASE 9 — STEP 9.6: VALIDATION-BASED MODEL SELECTION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Define saved candidate and result paths
# ------------------------------------------------------------

PROJECT_DIR = Path(r"D:\smart-cybersecurity-ml")
RESULTS_DIR = PROJECT_DIR / "results" / "phase9"
MODELS_DIR = PROJECT_DIR / "models"

comparison_path = (
    RESULTS_DIR / "phase9_step9_5_validation_comparison.csv"
)

baseline_path = (
    MODELS_DIR / "phase9_baseline_xgboost_candidate.pkl"
)

weighted_path = (
    MODELS_DIR / "phase9_weighted_xgboost_candidate.pkl"
)

# ------------------------------------------------------------
# 2. Load validation results
# ------------------------------------------------------------

comparison_df = pd.read_csv(comparison_path, index_col=0)

required_metrics = [
    "macro_f1",
    "balanced_accuracy",
    "macro_recall"
]

missing_metrics = [
    metric for metric in required_metrics
    if metric not in comparison_df.columns
]

assert not missing_metrics, (
    f"Missing required validation metrics: {missing_metrics}"
)

assert "Baseline XGBoost" in comparison_df.index
assert "Class-Weighted XGBoost" in comparison_df.index

# ------------------------------------------------------------
# 3. Select the best candidate
# Primary metric: macro F1
# Tie-breaker: balanced accuracy
# ------------------------------------------------------------

ranked_results = comparison_df.sort_values(
    by=["macro_f1", "balanced_accuracy"],
    ascending=False
)

selected_name = ranked_results.index[0]

candidate_paths = {
    "Baseline XGBoost": baseline_path,
    "Class-Weighted XGBoost": weighted_path
}

selected_path = candidate_paths[selected_name]

assert selected_path.exists(), (
    f"Selected candidate model not found: {selected_path}"
)

# Load the selected model and save a separate selection artifact.
selected_model = joblib.load(selected_path)

selected_model_path = (
    MODELS_DIR / "phase9_selected_xgboost_candidate.pkl"
)

joblib.dump(selected_model, selected_model_path)

# ------------------------------------------------------------
# 4. Save the selection report
# ------------------------------------------------------------

selected_metrics = {
    metric: float(comparison_df.loc[selected_name, metric])
    for metric in required_metrics
}

selection_report = {
    "phase": 9,
    "step": "9.6",
    "selection_method": (
        "Highest validation macro F1; "
        "balanced accuracy used as tie-breaker"
    ),
    "selected_candidate": selected_name,
    "selected_metrics": selected_metrics,
    "selected_model_path": str(selected_model_path),
    "final_test_set_used_for_selection": False,
    "status": "Selected for further validation"
}

report_path = (
    RESULTS_DIR / "phase9_step9_6_model_selection.json"
)

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(selection_report, file, indent=4)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\n--- Candidate Comparison ---")
print(
    comparison_df[
        ["accuracy", "balanced_accuracy", "macro_f1"]
    ].round(4).to_string()
)

print("\n--- Selection Result ---")
print("Selected candidate:", selected_name)
print("Selection metric: Validation macro F1")
print("Selected macro F1:", round(selected_metrics["macro_f1"], 4))
print(
    "Selected balanced accuracy:",
    round(selected_metrics["balanced_accuracy"], 4)
)

print("\nSelected candidate saved to:")
print(selected_model_path)

print("\nSelection report saved to:")
print(report_path)

print("\nFinal test set used for selection: No")
print("Step 9.6 completed.")


PHASE 9 — STEP 9.6: VALIDATION-BASED MODEL SELECTION

--- Candidate Comparison ---
                        accuracy  balanced_accuracy  macro_f1
Baseline XGBoost          0.9974             0.7938    0.8267
Class-Weighted XGBoost    0.9964             0.9057    0.8869

--- Selection Result ---
Selected candidate: Class-Weighted XGBoost
Selection metric: Validation macro F1
Selected macro F1: 0.8869
Selected balanced accuracy: 0.9057

Selected candidate saved to:
D:\smart-cybersecurity-ml\models\phase9_selected_xgboost_candidate.pkl

Selection report saved to:
D:\smart-cybersecurity-ml\results\phase9\phase9_step9_6_model_selection.json

Final test set used for selection: No
Step 9.6 completed.


## Step 9.7 — Selected Model Error Analysis
### Validation Confusion Matrix and Misclassified Samples

In [13]:

# ============================================================
# PHASE 9 — STEP 9.7: SELECTED MODEL ERROR ANALYSIS
# Validation Confusion Matrix and Misclassified Samples
# ============================================================

import numpy as np
import pandas as pd
import joblib
import json

from pathlib import Path
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support
)

print("=" * 75)
print("PHASE 9 — STEP 9.7: SELECTED MODEL ERROR ANALYSIS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Define project paths and load selected model
# ------------------------------------------------------------

PROJECT_DIR = Path(r"D:\smart-cybersecurity-ml")
RESULTS_DIR = PROJECT_DIR / "results" / "phase9"
MODELS_DIR = PROJECT_DIR / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

selected_model_path = (
    MODELS_DIR / "phase9_selected_xgboost_candidate.pkl"
)

selected_model = joblib.load(selected_model_path)

# ------------------------------------------------------------
# 2. Verify validation data from Step 9.5
# ------------------------------------------------------------

assert "X_val" in globals(), (
    "X_val is missing. Run Step 9.5 first to recreate the validation split."
)
assert "y_val" in globals(), (
    "y_val is missing. Run Step 9.5 first to recreate the validation split."
)
assert len(X_val) == len(y_val), "Validation feature/label lengths differ."

print("\n--- Validation Dataset ---")
print("Validation samples:", len(X_val))
print("Validation features:", X_val.shape[1])
print("Selected model:", selected_model_path.name)

# ------------------------------------------------------------
# 3. Generate validation predictions
# ------------------------------------------------------------

y_val_true = np.asarray(y_val).ravel()
y_val_pred = selected_model.predict(X_val)

print("\n--- Overall Validation Metrics ---")
print(f"Accuracy: {accuracy_score(y_val_true, y_val_pred):.4f}")
print(
    "Balanced accuracy: "
    f"{balanced_accuracy_score(y_val_true, y_val_pred):.4f}"
)

# ------------------------------------------------------------
# 4. Create class-wise error analysis
# ------------------------------------------------------------

class_ids = np.arange(len(target_encoder.classes_))
class_names = target_encoder.classes_

precision, recall, f1, support = precision_recall_fscore_support(
    y_val_true,
    y_val_pred,
    labels=class_ids,
    zero_division=0
)

classwise_df = pd.DataFrame({
    "class_id": class_ids,
    "class_name": class_names,
    "support": support,
    "precision": precision,
    "recall": recall,
    "f1_score": f1
})

classwise_df["correct_samples"] = np.rint(
    classwise_df["recall"] * classwise_df["support"]
).astype(int)

classwise_df["misclassified_samples"] = (
    classwise_df["support"] - classwise_df["correct_samples"]
)

classwise_path = (
    RESULTS_DIR / "phase9_step9_7_classwise_errors.csv"
)

classwise_df.to_csv(classwise_path, index=False)

print("\n--- Lowest Recall Classes ---")
print(
    classwise_df.sort_values(
        ["recall", "support"],
        ascending=[True, True]
    ).head(10).round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Identify misclassified validation samples
# ------------------------------------------------------------

misclassified_mask = y_val_true != y_val_pred

misclassified_df = pd.DataFrame({
    "validation_row_index": np.asarray(X_val.index),
    "actual_class_id": y_val_true,
    "predicted_class_id": y_val_pred,
    "actual_class": class_names[y_val_true.astype(int)],
    "predicted_class": class_names[y_val_pred.astype(int)]
})

misclassified_df = misclassified_df.loc[
    misclassified_mask
].copy()

misclassified_path = (
    RESULTS_DIR / "phase9_step9_7_misclassified_samples.csv"
)

misclassified_df.to_csv(misclassified_path, index=False)

print("\n--- Misclassification Summary ---")
print("Total validation samples:", len(y_val_true))
print("Correct predictions:", int((~misclassified_mask).sum()))
print("Incorrect predictions:", int(misclassified_mask.sum()))
print(
    "Validation error rate:",
    round(float(misclassified_mask.mean()), 6)
)

# ------------------------------------------------------------
# 6. Build confusion matrix and save as CSV
# ------------------------------------------------------------

cm = confusion_matrix(
    y_val_true,
    y_val_pred,
    labels=class_ids
)

confusion_df = pd.DataFrame(
    cm,
    index=[f"Actual_{name}" for name in class_names],
    columns=[f"Predicted_{name}" for name in class_names]
)

confusion_path = (
    RESULTS_DIR / "phase9_step9_7_confusion_matrix.csv"
)

confusion_df.to_csv(confusion_path)

# ------------------------------------------------------------
# 7. Summarize the most common class confusions
# ------------------------------------------------------------

confusion_pairs = []

for actual_id in class_ids:
    for predicted_id in class_ids:
        if actual_id != predicted_id and cm[actual_id, predicted_id] > 0:
            confusion_pairs.append({
                "actual_class": class_names[actual_id],
                "predicted_class": class_names[predicted_id],
                "misclassified_samples": int(cm[actual_id, predicted_id])
            })

confusion_pairs_df = pd.DataFrame(
    confusion_pairs,
    columns=[
        "actual_class",
        "predicted_class",
        "misclassified_samples"
    ]
)

if not confusion_pairs_df.empty:
    confusion_pairs_df = confusion_pairs_df.sort_values(
        "misclassified_samples",
        ascending=False
    )

confusion_pairs_path = (
    RESULTS_DIR / "phase9_step9_7_top_confusions.csv"
)

confusion_pairs_df.to_csv(confusion_pairs_path, index=False)

print("\n--- Most Common Class Confusions ---")

if confusion_pairs_df.empty:
    print("No class confusions found on the validation set.")
else:
    print(confusion_pairs_df.head(15).to_string(index=False))

# ------------------------------------------------------------
# 8. Save a JSON summary
# ------------------------------------------------------------

summary = {
    "step": "9.7",
    "model": "Class-Weighted XGBoost",
    "validation_samples": int(len(y_val_true)),
    "correct_predictions": int((~misclassified_mask).sum()),
    "incorrect_predictions": int(misclassified_mask.sum()),
    "accuracy": float(accuracy_score(y_val_true, y_val_pred)),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_val_true, y_val_pred)
    ),
    "final_test_set_used": False,
    "status": "Validation error analysis completed"
}

summary_path = (
    RESULTS_DIR / "phase9_step9_7_error_analysis_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# ------------------------------------------------------------
# 9. Display saved artifacts
# ------------------------------------------------------------

print("\n--- Saved Files ---")
print(classwise_path)
print(misclassified_path)
print(confusion_path)
print(confusion_pairs_path)
print(summary_path)

print("\nFinal test set used: No")
print("Step 9.7 completed.")


PHASE 9 — STEP 9.7: SELECTED MODEL ERROR ANALYSIS

--- Validation Dataset ---
Validation samples: 23757
Validation features: 41
Selected model: phase9_selected_xgboost_candidate.pkl

--- Overall Validation Metrics ---
Accuracy: 0.9964
Balanced accuracy: 0.9057

--- Lowest Recall Classes ---
 class_id      class_name  support  precision  recall  f1_score  correct_samples  misclassified_samples
       19         rootkit        2     0.0000  0.0000    0.0000                0                      2
        6            imap        2     1.0000  0.5000    0.6667                1                      1
        2 buffer_overflow        8     1.0000  0.6250    0.7692                5                      3
       12        multihop        3     1.0000  0.6667    0.8000                2                      1
       20           saint       15     0.8571  0.8000    0.8276               12                      3
       23   snmpgetattack        7     0.3333  0.8571    0.4800                6    

## Step 9.8 — Class Imbalance Strategy Evaluation
### Compare Baseline and Class-Weighted XGBoost

In [14]:

# ============================================================
# PHASE 9 — STEP 9.8: CLASS IMBALANCE STRATEGY EVALUATION
# Compare Baseline and Class-Weighted XGBoost
# ============================================================

import numpy as np
import pandas as pd
import joblib
import json

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

print("=" * 75)
print("PHASE 9 — STEP 9.8: CLASS IMBALANCE STRATEGY EVALUATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Set project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(r"D:\smart-cybersecurity-ml")
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results" / "phase9"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Load both saved candidate models
# ------------------------------------------------------------

baseline_model = joblib.load(
    MODELS_DIR / "phase9_baseline_xgboost_candidate.pkl"
)

weighted_model = joblib.load(
    MODELS_DIR / "phase9_weighted_xgboost_candidate.pkl"
)

# ------------------------------------------------------------
# 3. Verify the existing validation split
# ------------------------------------------------------------

assert "X_val" in globals(), (
    "X_val is missing. Run Step 9.5 first."
)
assert "y_val" in globals(), (
    "y_val is missing. Run Step 9.5 first."
)
assert len(X_val) == len(y_val), (
    "Validation features and labels have different lengths."
)

y_true = np.asarray(y_val).ravel().astype(int)

class_names = np.asarray(target_encoder.classes_)
class_ids = np.arange(len(class_names))

print("\n--- Validation Data ---")
print("Validation samples:", len(y_true))
print("Features:", X_val.shape[1])
print("Number of classes:", len(class_names))

# ------------------------------------------------------------
# 4. Generate predictions on the SAME validation set
# ------------------------------------------------------------

baseline_pred = np.asarray(
    baseline_model.predict(X_val)
).ravel().astype(int)

weighted_pred = np.asarray(
    weighted_model.predict(X_val)
).ravel().astype(int)

# ------------------------------------------------------------
# 5. Calculate overall metrics
# ------------------------------------------------------------

def calculate_metrics(y_actual, y_predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_actual,
        y_predicted,
        labels=class_ids,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(y_actual, y_predicted),
        "balanced_accuracy": balanced_accuracy_score(
            y_actual, y_predicted
        ),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }


baseline_metrics = calculate_metrics(y_true, baseline_pred)
weighted_metrics = calculate_metrics(y_true, weighted_pred)

comparison_df = pd.DataFrame([
    {"model": "Baseline XGBoost", **baseline_metrics},
    {"model": "Class-Weighted XGBoost", **weighted_metrics}
])

print("\n--- Overall Validation Comparison ---")
print(comparison_df.round(4).to_string(index=False))

# ------------------------------------------------------------
# 6. Compare class-wise precision, recall and F1
# ------------------------------------------------------------

def make_class_metrics(y_actual, y_predicted, model_name):
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_actual,
            y_predicted,
            labels=class_ids,
            zero_division=0
        )
    )

    return pd.DataFrame({
        "class_id": class_ids,
        "class_name": class_names,
        "support": support,
        f"{model_name}_precision": precision,
        f"{model_name}_recall": recall,
        f"{model_name}_f1": f1
    })


baseline_class_df = make_class_metrics(
    y_true, baseline_pred, "baseline"
)

weighted_class_df = make_class_metrics(
    y_true, weighted_pred, "weighted"
)

class_comparison_df = baseline_class_df.merge(
    weighted_class_df[
        [
            "class_id",
            "weighted_precision",
            "weighted_recall",
            "weighted_f1"
        ]
    ],
    on="class_id",
    how="left"
)

class_comparison_df["recall_change"] = (
    class_comparison_df["weighted_recall"]
    - class_comparison_df["baseline_recall"]
)

class_comparison_df["f1_change"] = (
    class_comparison_df["weighted_f1"]
    - class_comparison_df["baseline_f1"]
)

class_comparison_path = (
    RESULTS_DIR / "phase9_step9_8_class_comparison.csv"
)

class_comparison_df.to_csv(
    class_comparison_path,
    index=False
)

# ------------------------------------------------------------
# 7. Inspect rare classes with fewer than 50 validation samples
# ------------------------------------------------------------

rare_classes_df = class_comparison_df[
    class_comparison_df["support"] < 50
].copy()

print("\n--- Rare-Class Comparison (Support < 50) ---")

print(
    rare_classes_df[
        [
            "class_name",
            "support",
            "baseline_recall",
            "weighted_recall",
            "baseline_f1",
            "weighted_f1",
            "recall_change"
        ]
    ].round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 8. Compare false positives for NORMAL traffic
# ------------------------------------------------------------

normal_matches = np.flatnonzero(class_names == "normal")

if len(normal_matches) == 0:
    raise ValueError(
        "The class 'normal' was not found in target_encoder.classes_."
    )

normal_id = int(normal_matches[0])
actual_normal_mask = y_true == normal_id

baseline_normal_fp = int(
    np.sum(
        actual_normal_mask
        & (baseline_pred != normal_id)
    )
)

weighted_normal_fp = int(
    np.sum(
        actual_normal_mask
        & (weighted_pred != normal_id)
    )
)

normal_count = int(actual_normal_mask.sum())

false_positive_df = pd.DataFrame([
    {
        "model": "Baseline XGBoost",
        "actual_normal_samples": normal_count,
        "normal_samples_misclassified_as_attacks": baseline_normal_fp,
        "normal_false_positive_rate": (
            baseline_normal_fp / normal_count
            if normal_count else 0.0
        )
    },
    {
        "model": "Class-Weighted XGBoost",
        "actual_normal_samples": normal_count,
        "normal_samples_misclassified_as_attacks": weighted_normal_fp,
        "normal_false_positive_rate": (
            weighted_normal_fp / normal_count
            if normal_count else 0.0
        )
    }
])

print("\n--- Normal Traffic False-Positive Comparison ---")
print(false_positive_df.round(6).to_string(index=False))

# ------------------------------------------------------------
# 9. Save confusion matrices for both candidates
# ------------------------------------------------------------

baseline_cm = confusion_matrix(
    y_true, baseline_pred, labels=class_ids
)

weighted_cm = confusion_matrix(
    y_true, weighted_pred, labels=class_ids
)

labels = [f"Actual_{name}" for name in class_names]
columns = [f"Predicted_{name}" for name in class_names]

pd.DataFrame(
    baseline_cm,
    index=labels,
    columns=columns
).to_csv(
    RESULTS_DIR / "phase9_step9_8_baseline_confusion_matrix.csv"
)

pd.DataFrame(
    weighted_cm,
    index=labels,
    columns=columns
).to_csv(
    RESULTS_DIR / "phase9_step9_8_weighted_confusion_matrix.csv"
)

# ------------------------------------------------------------
# 10. Save overall comparison and summary
# ------------------------------------------------------------

comparison_path = (
    RESULTS_DIR / "phase9_step9_8_overall_comparison.csv"
)

comparison_df.to_csv(comparison_path, index=False)

summary = {
    "step": "9.8",
    "validation_samples": int(len(y_true)),
    "baseline_metrics": {
        key: float(value)
        for key, value in baseline_metrics.items()
    },
    "weighted_metrics": {
        key: float(value)
        for key, value in weighted_metrics.items()
    },
    "normal_samples_misclassified_by_baseline": baseline_normal_fp,
    "normal_samples_misclassified_by_weighted": weighted_normal_fp,
    "final_test_set_used": False,
    "status": "Class imbalance strategy comparison completed"
}

summary_path = (
    RESULTS_DIR / "phase9_step9_8_comparison_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# ------------------------------------------------------------
# 11. Final output
# ------------------------------------------------------------

print("\n--- Saved Files ---")
print(comparison_path)
print(class_comparison_path)
print(
    RESULTS_DIR / "phase9_step9_8_baseline_confusion_matrix.csv"
)
print(
    RESULTS_DIR / "phase9_step9_8_weighted_confusion_matrix.csv"
)
print(summary_path)

print("\nFinal test set used: No")
print("Step 9.8 completed.")


PHASE 9 — STEP 9.8: CLASS IMBALANCE STRATEGY EVALUATION

--- Validation Data ---
Validation samples: 23757
Features: 41
Number of classes: 28

--- Overall Validation Comparison ---
                 model  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1
      Baseline XGBoost    0.9974             0.7938           0.8945        0.7938    0.8267
Class-Weighted XGBoost    0.9964             0.9057           0.9001        0.9057    0.8869

--- Rare-Class Comparison (Support < 50) ---
     class_name  support  baseline_recall  weighted_recall  baseline_f1  weighted_f1  recall_change
        apache2       36           1.0000           1.0000       1.0000       1.0000         0.0000
buffer_overflow        8           0.5000           0.6250       0.6667       0.7692         0.1250
      ftp_write        1           1.0000           1.0000       1.0000       1.0000         0.0000
     httptunnel        7           0.7143           1.0000       0.8333       1.0000         0

Phase 9 — Step 9.9: Threshold and Alert Policy Analysis

In [15]:

# ============================================================
# PHASE 9 — STEP 9.9
# THRESHOLD AND ALERT POLICY ANALYSIS
# Validation Data Only — No Final Test Set
# ============================================================

import numpy as np
import pandas as pd
import joblib
import json

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

print("=" * 75)
print("PHASE 9 — STEP 9.9: THRESHOLD AND ALERT POLICY ANALYSIS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Define paths and load selected model
# ------------------------------------------------------------

PROJECT_DIR = Path(r"D:\smart-cybersecurity-ml")
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results" / "phase9"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODELS_DIR / "phase9_selected_xgboost_candidate.pkl"
selected_model = joblib.load(model_path)

# ------------------------------------------------------------
# 2. Verify existing validation data
# ------------------------------------------------------------

assert "X_val" in globals(), (
    "X_val is missing. Run Step 9.5 first."
)
assert "y_val" in globals(), (
    "y_val is missing. Run Step 9.5 first."
)
assert len(X_val) == len(y_val), (
    "Validation feature and label counts do not match."
)

y_true = np.asarray(y_val).ravel().astype(int)
class_names = np.asarray(target_encoder.classes_)
class_ids = np.arange(len(class_names))

print("\n--- Validation Data ---")
print("Validation samples:", len(y_true))
print("Number of features:", X_val.shape[1])
print("Number of classes:", len(class_names))

# ------------------------------------------------------------
# 3. Generate probabilities
# ------------------------------------------------------------

if not hasattr(selected_model, "predict_proba"):
    raise TypeError(
        "Selected model does not support predict_proba()."
    )

probabilities = selected_model.predict_proba(X_val)

if probabilities.shape[1] != len(class_names):
    raise ValueError(
        "Model probability columns do not match target encoder classes. "
        "Check the model's classes_ ordering before continuing."
    )

predicted_class_ids = np.argmax(probabilities, axis=1)
max_confidence = np.max(probabilities, axis=1)

print("\n--- Confidence Summary ---")
print(f"Mean confidence: {max_confidence.mean():.4f}")
print(f"Median confidence: {np.median(max_confidence):.4f}")
print(f"Minimum confidence: {max_confidence.min():.4f}")
print(f"Maximum confidence: {max_confidence.max():.4f}")

# ------------------------------------------------------------
# 4. Confidence analysis for correct vs incorrect predictions
# ------------------------------------------------------------

correct_mask = predicted_class_ids == y_true
incorrect_mask = ~correct_mask

confidence_summary = pd.DataFrame([
    {
        "prediction_group": "Correct",
        "sample_count": int(correct_mask.sum()),
        "mean_confidence": (
            float(max_confidence[correct_mask].mean())
            if correct_mask.any() else None
        )
    },
    {
        "prediction_group": "Incorrect",
        "sample_count": int(incorrect_mask.sum()),
        "mean_confidence": (
            float(max_confidence[incorrect_mask].mean())
            if incorrect_mask.any() else None
        )
    }
])

print("\n--- Confidence: Correct vs Incorrect ---")
print(confidence_summary.round(4).to_string(index=False))

confidence_summary.to_csv(
    RESULTS_DIR / "phase9_step9_9_confidence_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 5. Define illustrative alert thresholds
# ------------------------------------------------------------
# IMPORTANT:
# These thresholds are experimental policy settings, not calibrated
# probabilities and not proof that a prediction is correct.
#
# ALERT: predicted class is an attack and confidence >= threshold
# REVIEW: predicted class is an attack but confidence < threshold
# ALLOW: predicted class is normal and confidence >= threshold
# REVIEW: predicted class is normal but confidence < threshold
#
# This is a simulation only. No real network traffic is blocked.

thresholds = [0.50, 0.70, 0.90, 0.95, 0.99]

normal_matches = np.flatnonzero(class_names == "normal")

if len(normal_matches) == 0:
    raise ValueError(
        "The target encoder does not contain the 'normal' class."
    )

normal_id = int(normal_matches[0])
predicted_is_normal = predicted_class_ids == normal_id
actual_is_normal = y_true == normal_id

threshold_results = []

for threshold in thresholds:
    high_confidence = max_confidence >= threshold

    # A high-confidence normal prediction is ALLOW.
    # A high-confidence attack prediction is ALERT.
    # Low-confidence predictions are REVIEW.
    decisions = np.full(len(y_true), "REVIEW", dtype=object)

    decisions[predicted_is_normal & high_confidence] = "ALLOW"
    decisions[(~predicted_is_normal) & high_confidence] = "ALERT"

    allow_mask = decisions == "ALLOW"
    alert_mask = decisions == "ALERT"
    review_mask = decisions == "REVIEW"

    # Correctly detected attacks = actual attack predicted as an
    # attack with sufficient confidence.
    detected_attack_mask = (
        (~actual_is_normal)
        & (~predicted_is_normal)
        & high_confidence
    )

    actual_attack_count = int((~actual_is_normal).sum())

    attack_detection_rate = (
        detected_attack_mask.sum() / actual_attack_count
        if actual_attack_count else 0.0
    )

    # Normal traffic incorrectly flagged as an attack.
    false_positive_mask = actual_is_normal & alert_mask

    normal_count = int(actual_is_normal.sum())

    false_positive_rate = (
        false_positive_mask.sum() / normal_count
        if normal_count else 0.0
    )

    threshold_results.append({
        "threshold": threshold,
        "allow_count": int(allow_mask.sum()),
        "alert_count": int(alert_mask.sum()),
        "review_count": int(review_mask.sum()),
        "attack_detection_rate": float(attack_detection_rate),
        "normal_false_positive_count": int(
            false_positive_mask.sum()
        ),
        "normal_false_positive_rate": float(false_positive_rate)
    })

threshold_df = pd.DataFrame(threshold_results)

print("\n--- Threshold and Alert Policy Comparison ---")
print(threshold_df.round(4).to_string(index=False))

threshold_path = (
    RESULTS_DIR / "phase9_step9_9_threshold_comparison.csv"
)

threshold_df.to_csv(threshold_path, index=False)

# ------------------------------------------------------------
# 6. Save validation predictions and confidence scores
# ------------------------------------------------------------

prediction_df = pd.DataFrame({
    "validation_row_index": np.asarray(X_val.index),
    "actual_class_id": y_true,
    "predicted_class_id": predicted_class_ids,
    "actual_class": class_names[y_true],
    "predicted_class": class_names[predicted_class_ids],
    "max_confidence": max_confidence,
    "prediction_correct": correct_mask
})

prediction_path = (
    RESULTS_DIR / "phase9_step9_9_validation_predictions.csv"
)

prediction_df.to_csv(prediction_path, index=False)

# ------------------------------------------------------------
# 7. Evaluate the default argmax classification
# ------------------------------------------------------------

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    predicted_class_ids,
    labels=class_ids,
    average="macro",
    zero_division=0
)

default_metrics = {
    "accuracy": float(
        accuracy_score(y_true, predicted_class_ids)
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(y_true, predicted_class_ids)
    ),
    "macro_precision": float(precision),
    "macro_recall": float(recall),
    "macro_f1": float(f1)
}

print("\n--- Default Classification Metrics ---")
for metric_name, metric_value in default_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

# ------------------------------------------------------------
# 8. Save summary report
# ------------------------------------------------------------

summary = {
    "step": "9.9",
    "analysis": "Threshold and alert policy simulation",
    "model": "Class-Weighted XGBoost",
    "validation_samples": int(len(y_true)),
    "mean_confidence": float(max_confidence.mean()),
    "correct_predictions": int(correct_mask.sum()),
    "incorrect_predictions": int(incorrect_mask.sum()),
    "default_metrics": default_metrics,
    "thresholds_evaluated": thresholds,
    "final_test_set_used": False,
    "real_network_traffic_blocked": False,
    "status": "Threshold and alert policy analysis completed"
}

summary_path = (
    RESULTS_DIR / "phase9_step9_9_alert_policy_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# ------------------------------------------------------------
# 9. Display saved artifacts
# ------------------------------------------------------------

print("\n--- Saved Files ---")
print(RESULTS_DIR / "phase9_step9_9_confidence_summary.csv")
print(threshold_path)
print(prediction_path)
print(summary_path)

print("\nFinal test set used: No")
print("Real network traffic blocked: No")
print("Step 9.9 completed.")


PHASE 9 — STEP 9.9: THRESHOLD AND ALERT POLICY ANALYSIS

--- Validation Data ---
Validation samples: 23757
Number of features: 41
Number of classes: 28

--- Confidence Summary ---
Mean confidence: 0.9967
Median confidence: 0.9999
Minimum confidence: 0.4171
Maximum confidence: 1.0000

--- Confidence: Correct vs Incorrect ---
prediction_group  sample_count  mean_confidence
         Correct         23671           0.9973
       Incorrect            86           0.8137

--- Threshold and Alert Policy Comparison ---
 threshold  allow_count  alert_count  review_count  attack_detection_rate  normal_false_positive_count  normal_false_positive_rate
      0.50        12534        11215             8                 0.9985                           49                      0.0039
      0.70        12509        11192            56                 0.9976                           36                      0.0029
      0.90        12415        11152           190                 0.9951                 

## Step 9.10 — Final Phase 9 Report
### Consolidated Results, Findings and Limitations

In [16]:

# ============================================================
# PHASE 9 — STEP 9.10: FINAL PHASE 9 REPORT
# Consolidate Steps 9.1 to 9.9
# ============================================================

import json
import pandas as pd

from pathlib import Path
from datetime import datetime

print("=" * 75)
print("PHASE 9 — STEP 9.10: FINAL PHASE 9 REPORT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Define project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(r"D:\smart-cybersecurity-ml")
RESULTS_DIR = PROJECT_DIR / "results" / "phase9"
REPORTS_DIR = PROJECT_DIR / "reports" / "phase9"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

report_path = REPORTS_DIR / "phase9_final_report.md"
summary_path = RESULTS_DIR / "phase9_step9_10_final_summary.json"

# ------------------------------------------------------------
# 2. Load available result files safely
# ------------------------------------------------------------

def load_csv(filename):
    path = RESULTS_DIR / filename
    if path.exists():
        return pd.read_csv(path)
    print(f"Warning: Missing file: {path}")
    return None


def load_json(filename):
    path = RESULTS_DIR / filename
    if path.exists():
        with open(path, "r", encoding="utf-8") as file:
            return json.load(file)
    print(f"Warning: Missing file: {path}")
    return None


validation_comparison = load_csv(
    "phase9_step9_5_validation_comparison.csv"
)

model_selection = load_json(
    "phase9_step9_6_model_selection.json"
)

classwise_errors = load_csv(
    "phase9_step9_7_classwise_errors.csv"
)

overall_comparison = load_csv(
    "phase9_step9_8_overall_comparison.csv"
)

class_comparison = load_csv(
    "phase9_step9_8_class_comparison.csv"
)

threshold_comparison = load_csv(
    "phase9_step9_9_threshold_comparison.csv"
)

confidence_summary = load_csv(
    "phase9_step9_9_confidence_summary.csv"
)

alert_summary = load_json(
    "phase9_step9_9_alert_policy_summary.json"
)

# ------------------------------------------------------------
# 3. Prepare report sections
# ------------------------------------------------------------

report_sections = []

report_sections.append("# Phase 9 — Model Evaluation and Improvement Report")
report_sections.append("")
report_sections.append(
    f"Report generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)
report_sections.append("")
report_sections.append("## 1. Objective")
report_sections.append(
    "Phase 9 evaluates class imbalance handling, compares baseline and "
    "class-weighted XGBoost, investigates validation errors, and studies "
    "confidence-based alert policies."
)
report_sections.append("")
report_sections.append("## 2. Model Selection")
report_sections.append(
    "The selected candidate was chosen using validation macro F1, with "
    "balanced accuracy as a supporting metric. The final test set was not "
    "used to select the Step 9.5 candidate."
)
report_sections.append("")

if validation_comparison is not None:
    report_sections.append("### Step 9.5 — Validation Comparison")
    report_sections.append("")
    report_sections.append(
        validation_comparison.to_string(index=False)
    )
    report_sections.append("")

if model_selection is not None:
    report_sections.append("### Step 9.6 — Selection Record")
    report_sections.append("")
    report_sections.append("```json")
    report_sections.append(json.dumps(model_selection, indent=2))
    report_sections.append("```")
    report_sections.append("")

# ------------------------------------------------------------
# 4. Report error analysis
# ------------------------------------------------------------

report_sections.append("## 3. Validation Error Analysis")
report_sections.append("")

if classwise_errors is not None:
    required_columns = {
        "class_name", "support", "precision", "recall", "f1_score"
    }

    if required_columns.issubset(classwise_errors.columns):
        lowest_recall = classwise_errors.sort_values(
            ["recall", "support"],
            ascending=[True, True]
        ).head(10)

        report_sections.append("### Classes with Lowest Recall")
        report_sections.append("")
        report_sections.append(
            lowest_recall[
                ["class_name", "support", "precision", "recall", "f1_score"]
            ].to_string(index=False)
        )
        report_sections.append("")

# ------------------------------------------------------------
# 5. Report class imbalance strategy comparison
# ------------------------------------------------------------

report_sections.append("## 4. Class Imbalance Strategy Evaluation")
report_sections.append("")

if overall_comparison is not None:
    report_sections.append(
        overall_comparison.to_string(index=False)
    )
    report_sections.append("")

if class_comparison is not None:
    if {
        "class_name",
        "support",
        "baseline_recall",
        "weighted_recall",
        "baseline_f1",
        "weighted_f1"
    }.issubset(class_comparison.columns):

        rare_classes = class_comparison[
            class_comparison["support"] < 50
        ].copy()

        report_sections.append("### Rare-Class Comparison (Support < 50)")
        report_sections.append("")
        report_sections.append(
            rare_classes[
                [
                    "class_name",
                    "support",
                    "baseline_recall",
                    "weighted_recall",
                    "baseline_f1",
                    "weighted_f1"
                ]
            ].to_string(index=False)
        )
        report_sections.append("")

# ------------------------------------------------------------
# 6. Report confidence and threshold analysis
# ------------------------------------------------------------

report_sections.append("## 5. Confidence and Alert Policy Analysis")
report_sections.append("")

if confidence_summary is not None:
    report_sections.append("### Correct vs Incorrect Predictions")
    report_sections.append("")
    report_sections.append(
        confidence_summary.to_string(index=False)
    )
    report_sections.append("")

if threshold_comparison is not None:
    report_sections.append("### Threshold Comparison")
    report_sections.append("")
    report_sections.append(
        threshold_comparison.to_string(index=False)
    )
    report_sections.append("")

if alert_summary is not None:
    report_sections.append("### Alert Policy Summary")
    report_sections.append("")
    report_sections.append("```json")
    report_sections.append(json.dumps(alert_summary, indent=2))
    report_sections.append("```")
    report_sections.append("")

# ------------------------------------------------------------
# 7. Findings and limitations
# ------------------------------------------------------------

report_sections.append("## 6. Main Findings")
report_sections.append("")
report_sections.append(
    "- Class-weighted XGBoost improved validation macro F1 and balanced "
    "accuracy compared with the baseline."
)
report_sections.append(
    "- Class weighting improved recall for several rare attacks, but did "
    "not solve every rare-class detection problem."
)
report_sections.append(
    "- The weighted model generated more false positives on normal traffic "
    "than the baseline in the reported validation comparison."
)
report_sections.append(
    "- Raising the alert confidence threshold reduced normal-traffic "
    "false positives, while also reducing the proportion of attacks "
    "flagged automatically and increasing review cases."
)
report_sections.append(
    "- Confidence scores are not necessarily calibrated probabilities "
    "and should not be treated as guaranteed measures of correctness."
)
report_sections.append("")

report_sections.append("## 7. Limitations")
report_sections.append("")
report_sections.append(
    "- Some rare classes have very few validation examples, so their "
    "precision and recall estimates are unstable."
)
report_sections.append(
    "- Duplicate feature patterns and eight previously identified "
    "overlapping rows with conflicting labels require careful "
    "interpretation; they do not by themselves prove data leakage."
)
report_sections.append(
    "- The final test set was evaluated during Phase 8. Therefore, although "
    "Steps 9.5–9.9 used validation data, the final test set should not be "
    "described as completely untouched by the overall project."
)
report_sections.append(
    "- The alert policy was simulated on validation predictions; no real "
    "network traffic was blocked."
)
report_sections.append(
    "- Results from the current dataset alone do not establish performance "
    "on CICIDS2017 or on real production traffic."
)
report_sections.append("")

report_sections.append("## 8. Conclusion")
report_sections.append("")
report_sections.append(
    "Class-weighted XGBoost is a promising candidate for improving "
    "minority-class performance on this validation split. Its higher "
    "false-positive count shows that a deployment policy must balance "
    "attack detection with alert workload. Further external validation "
    "and testing on additional datasets are needed before claiming "
    "production readiness."
)
report_sections.append("")

report_sections.append("## 9. Final Test Set Policy")
report_sections.append("")
report_sections.append(
    "Do not use the final test set for further threshold tuning or "
    "candidate selection. For stronger independent evidence, evaluate "
    "the finalized system on a new external dataset or a newly collected "
    "holdout set."
)

# ------------------------------------------------------------
# 8. Write the Markdown report
# ------------------------------------------------------------

report_text = "\n".join(report_sections)

with open(report_path, "w", encoding="utf-8") as file:
    file.write(report_text)

# ------------------------------------------------------------
# 9. Save a compact JSON summary
# ------------------------------------------------------------

summary = {
    "phase": 9,
    "status": "Report generated",
    "selected_model": "Class-Weighted XGBoost",
    "model_selection_basis": "Validation macro F1",
    "validation_accuracy": 0.9964,
    "validation_balanced_accuracy": 0.9057,
    "validation_macro_f1": 0.8869,
    "thresholds_evaluated": (
        threshold_comparison["threshold"].tolist()
        if threshold_comparison is not None
        and "threshold" in threshold_comparison.columns
        else []
    ),
    "final_test_set_used_for_phase9_selection": False,
    "final_test_set_previously_evaluated_in_phase8": True,
    "real_network_traffic_blocked": False,
    "limitations_documented": True,
    "report_path": str(report_path)
}

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# ------------------------------------------------------------
# 10. Display completion status
# ------------------------------------------------------------

print("\n--- Phase 9 Summary ---")
print("Selected model: Class-Weighted XGBoost")
print("Validation accuracy: 99.64%")
print("Validation balanced accuracy: 90.57%")
print("Validation macro F1: 88.69%")
print("Steps covered: 9.1 through 9.9")

print("\n--- Generated Files ---")
print("Final report:", report_path)
print("JSON summary:", summary_path)

print("\nFinal test set used for Step 9.10: No")
print("Step 9.10 completed.")


PHASE 9 — STEP 9.10: FINAL PHASE 9 REPORT

--- Phase 9 Summary ---
Selected model: Class-Weighted XGBoost
Validation accuracy: 99.64%
Validation balanced accuracy: 90.57%
Validation macro F1: 88.69%
Steps covered: 9.1 through 9.9

--- Generated Files ---
Final report: D:\smart-cybersecurity-ml\reports\phase9\phase9_final_report.md
JSON summary: D:\smart-cybersecurity-ml\results\phase9\phase9_step9_10_final_summary.json

Final test set used for Step 9.10: No
Step 9.10 completed.
